# Trajectory Container Tools - Dataclass Usage Example

This notebook demonstrates the essential usage of Trajectory Container Tools (TCT) dataclasses for creating and working with trajectory data.


## 1. Basic Setup

Import required modules and create sample data.


In [1]:
from dataclasses import dataclass
import numpy as np
from trajectory_container_tools.trj_dataclasses.base_trajectory_dataclass import BaseTrajectoryDataclass


## 2. Creating a Simple Trajectory

Create a basic trajectory with position data.


In [2]:
# Generate sample trajectory data
timesteps = 50
time_array = np.linspace(0, 5, timesteps)
x_positions = np.sin(time_array)
y_positions = np.cos(time_array)
timestamps = np.arange(timesteps) * 0.1  # 10 Hz sampling

@dataclass()
class Simple2DCoordinateTrajectory(BaseTrajectoryDataclass):
    x: np.ndarray
    y: np.ndarray
    timestamps: np.ndarray

# Create trajectory container
trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate",
    x=x_positions,
    y=y_positions,
    timestamps=timestamps
)

print(f"Created trajectory with {trajectory.trajectory_len} timesteps")
print(f"Available dimensions: {trajectory.get_dimension_names()}")
print(f"Data shapes: x={trajectory.x.shape}, y={trajectory.y.shape}")


Created trajectory with 50 timesteps
Available dimensions: ('x', 'y', 'timestamps')
Data shapes: x=(50,), y=(50,)


## 3. Accessing Trajectory Data

Demonstrate basic data access and manipulation.


In [3]:
# Access individual dimensions
x_data = trajectory.x
y_data = trajectory.y
time_data = trajectory.timestamps

print(f"X position range: [{x_data.min():.2f}, {x_data.max():.2f}]")
print(f"Y position range: [{y_data.min():.2f}, {y_data.max():.2f}]")
print(f"Time range: [{time_data.min():.2f}, {time_data.max():.2f}] seconds")


X position range: [-1.00, 1.00]
Y position range: [-1.00, 1.00]
Time range: [0.00, 4.90] seconds


## 4. Trajectory Slicing

Extract portions of the trajectory.


In [4]:
# Slice trajectory (get timesteps 10-30)
partial_trajectory = trajectory[10:30]
print(f"Original trajectory length: {trajectory.trajectory_len}")
print(f"Partial trajectory length: {partial_trajectory.trajectory_len}")

# Access sliced data
print(f"Partial X range: [{partial_trajectory.x.min():.2f}, {partial_trajectory.x.max():.2f}]")


Original trajectory length: 50
Partial trajectory length: 20
Partial X range: [0.18, 1.00]


## 5. Iterating Through Trajectory Points

Iterate through trajectory data points.


In [5]:
# Iterate through first 5 trajectory points
print("First 5 trajectory points:")
for i, point in enumerate(trajectory):
    if i >= 5:
        break
    print(f"Point {i}: x={point.x:.3f}, y={point.y:.3f}")


First 5 trajectory points:
Point 0: x=0.000, y=1.000
Point 1: x=0.102, y=0.995
Point 2: x=0.203, y=0.979
Point 3: x=0.301, y=0.954
Point 4: x=0.397, y=0.918


## 8. Data Validation

The dataclasses automatically validate data consistency.


In [6]:
# Example of data validation - this will work
valid_x = np.array([1, 2, 3, 4, 5])
valid_y = np.array([5, 4, 3, 2, 1])
valid_timestamps = np.array([0.0, 0.1, 0.2, 0.3, 0.4])


# Create trajectory container
valid_trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate – valid",
    x=valid_x,
    y=valid_y,
    timestamps=valid_timestamps
)

print("Valid trajectory created successfully!")
print(f"Trajectory length: {valid_trajectory.trajectory_len}")

# Demonstrate error handling with mismatched dimensions
try:
    # This should raise an error due to mismatched array lengths
    invalid_x = np.array([1, 2, 3])  # 3 elements
    invalid_y = np.array([1, 2, 3, 4])  # 4 elements - mismatch!
    invalid_timestamps = np.array([0.0, 0.1, 0.2])  # 3 elements

    invalid_trajectory = Simple2DCoordinateTrajectory(
        feature_name="2D coordinate – invalid",
        x=invalid_x,
        y=invalid_y,
        timestamps=invalid_timestamps
    )
    
except Exception as e:
    print(f"Expected error caught: {type(e).__name__}")
    print(f"Error message: {str(e)[:100]}...")


Valid trajectory created successfully!
Trajectory length: 5
Expected error caught: ValueError
Error message: 4 != 3
[TCT error] `2D coordinate – invalid` with container `y` received numpy arrays which do not m...


## 9. Trajectory batching example

In [7]:
# Create batch trajectories (3 trajectories, 20 timesteps each)
batch_size, time_steps = 3, 20
batch_x = np.random.randn(batch_size, time_steps)
batch_y = np.random.randn(batch_size, time_steps)
batch_frame = np.random.randn(batch_size, time_steps, 10)
batch_timestamps = np.tile(np.arange(time_steps) * 0.1, (batch_size, 1))

@dataclass()
class Simple2DCoordinateTrajectory(BaseTrajectoryDataclass):
    x: np.ndarray
    frame: np.ndarray
    y: np.ndarray
    timestamps: np.ndarray


batch_trajectory = Simple2DCoordinateTrajectory(
    feature_name="batch 2d coordinate",
    x=batch_x,
    y=batch_y,
    frame=batch_frame,
    timestamps=batch_timestamps,
    batch=True
)

print(f"Batch trajectory shape: {batch_trajectory.x.shape}")
print(f"Number of trajectories: {batch_trajectory.x.shape[0]}")
print(f"Timesteps per trajectory: {batch_trajectory.x.shape[1]}")
print(f"Trajectory length: {len(batch_trajectory)}")

print(batch_trajectory)

Batch trajectory shape: (3, 20)
Number of trajectories: 3
Timesteps per trajectory: 20
Trajectory length: 20

          Simple2DCoordinateTrajectory(
             feature_name: batch 2d coordinate
             trajectory_len: 20
             batch: True
             transposed: False
             dimensions:
                timesteps_indices: (ndarray) shape (20,) range 0 ⟶ 19
                x: (ndarray) shape (3, 20) range -1.741770832474461 ⟶ 1.6617714471425484
                frame: (ndarray) shape (3, 20, 10) range -2.908148975164058 ⟶ 3.4010544810470407
                y: (ndarray) shape (3, 20) range -1.5208578473935253 ⟶ 1.659558612175467
                timestamps: (ndarray) shape (3, 20) range 0.0 ⟶ 1.9000000000000001
          )


## Summary

This notebook covered the essential usage patterns of TCT dataclasses:

1. **Basic trajectory creation** with `BaseTrajectoryDataclass`
2. **Data access** and manipulation methods
3. **Trajectory slicing** for extracting portions of data
4. **Iteration** through trajectory points
5. **Reverse axis trajectories** for dataframe-like data
6. **Data validation** and error handling

For more advanced features and detailed examples, refer to the documentation at `documentation/direct_instantiation.md`.
